代码基于 HuggingFace TRL 0.18 的 GRPOTrainer 做增量改动即可——只把「先聚合再归一化」换成「每条奖励先单独归一化 → 再聚合 → 再做一次 batch 归一化」，其余训练流程不变。


## 安装依赖
- pip install transformers==4.40 accelerate datasets torch trl==0.18.0

## 代码（保存为 gdpo_trainer.py 可直接运行）

In [ ]:
import torch,os
from transformers import AutoTokenizer,AutoModelForCausalLM,TrainingArguments
from trl import GRPOTrainer
from datasets import Dataset

'''
基于TRL的GRPOTrainer做增量修改：
核心改动： advantage的计算部分-》先逐奖励归一化，再聚合，再batch归一化
'''

# 1. 构建偏好数据 每条样本 = {prompt,chosen,rejected}
preference_data = [
    {"prompt": "What is the capital of France?",
     "chosen": "The capital of France is Paris.",
     "rejected": "France’s capital is Berlin."},
    {"prompt": "Who painted the Mona Lisa?",
     "chosen": "Leonardo da Vinci painted the Mona Lisa.",
     "rejected": "The Mona Lisa was painted by Michelangelo."},
]
dataset = Dataset.from_list(preference_data)

# 2. 加载模型 & 分词器
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokennizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)

# 继承GRPOTrainer，仅重写advantage方法
class GDPOTrainer(GRPOTrainer):
    def compute_advantage(self,rewards_per_func):
        '''
        rewards_per_func:[B,nun_rewards] 每行是一个生成样本，每列式一个奖励行数打分
        返回 advantage：[B] 用于PPO-style的policy loss
        
        每行是一个生成样本（共B条)
        每列式一个奖励函数对该样本打分

        '''
        device = rewards_per_func.device
        num_generations = self.num_generations # 每个prompt生成多个输出
        reward_weights = self.reward_weights.to(device)
        
        # 去掉nan
        # 奖励函数可能返回NaN（解析失败等） 先清零避免后续爆炸
        r = torch.nan_to_num(reward_per_func)
        # 逐奖励单独归一化
        all_adv = []
        for i in range(len(reward_weights)):
            ri = r[:,i] # 取出第i种奖励
            meani = ri.view(-1,num_generations).mean(dim=1) # [B/num_gen]
            stdi = ri.view(-1,num_generations).std(dim=1)
            
            # 把均值、标准差广播回每条样本
            meani = meani.repeat_interleave(num_generations,dim=0)
            stdi = stdi.repeat_interleave(num_generaions,dim=0)
            adv_i = (ri - meani) / (stdi + 1e-4) # 零均值单位方差
            adll_adv.append(adv_i)
        combined = torch.stack(all_adv,dim=1) # [B,num_rewards]
        # 加权求和
        pre_bn = (combined * reward_weights.unsqueeze(0)).nansum(dim=1)
            
        # batch级归一化（保证全局均值单位方差）
        bn_mean = pre_bn.mean()
        bn_std = pre_bn.std()
        advantage = (pre_bn - bn_mean) / (bn_std + 1e-4)
        return advantages
        
# 4. 训练参数
training_args = TrainingArguments(
            output_dir = './gdpo_demo',
            per_device_train_batch_size = 2,
            num_train_epochs = 1,
            learing_rate = 1e-4,
            logging_steps = 1,
            report_to = 'none',
)
# 5. 启动训练
trainer = GDPOTrainer(
            model = model,
            ref_model = ref_model,
            args = training_args,
            train_dataset = dataset,
            tokenizer = tokenizer,
            beta = 0.1,
            num_generations = 4, # 每个prompt生成4个回答
            reward_weights = torch.tensor([0.5,0.5]), # 两条奖励函数，权重1:1
)
trainer.train(0)

## 运行


In [ ]:
python dapo_trainer.py

## 单奖励 group 归一化 → 加权求和 → batch 归一化，